# Preliminary Modeling

## Objectives
- Train preliminary models on both full and PCA-reduced features
    - Decision Tree Classifier
    - Random Forest
    - Gradient Boosting Classifier
- Evaluate models using Precision, Recall, and F1 Score

In [4]:
import pandas as pd

X_train = pd.read_csv("../data/X_train.csv")
X_train_pca = pd.read_csv("../data/X_train_pca.csv")
X_val = pd.read_csv("../data/X_val.csv")
X_val_pca = pd.read_csv("../data/X_val_pca.csv")
X_test = pd.read_csv("../data/X_test.csv")
X_test_pca = pd.read_csv("../data/X_test_pca.csv")

y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_val = pd.read_csv("../data/y_val.csv").squeeze()
y_test = pd.read_csv("../data/y_test.csv").squeeze()

print("X_train shape:", X_train.shape)
print("X_train_pca shape:", X_train_pca.shape)
print("X_val shape:", X_val.shape)
print("X_val_pca shape:", X_val_pca.shape)
print("X_test shape:", X_test.shape)
print("X_test_pca shape:", X_test_pca.shape)

print("\ny_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

X_train shape: (4025396, 8)
X_train_pca shape: (4025396, 7)
X_val shape: (437505, 8)
X_val_pca shape: (437505, 7)
X_test shape: (437505, 8)
X_test_pca shape: (437505, 7)

y_train shape: (4025396,)
y_val shape: (437505,)
y_test shape: (437505,)


In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

## Modeling

In [6]:
# Full-feature models
models_full = {
    "tree": DecisionTreeClassifier(random_state=12),
    "forest": RandomForestClassifier(n_jobs=-1, n_estimators=20, random_state=12),
    "grad": HistGradientBoostingClassifier(random_state=12)
}

# PCA-reduced models
models_pca = {
    "tree": DecisionTreeClassifier(random_state=12),
    "forest": RandomForestClassifier(n_jobs=-1, n_estimators=20, random_state=12),
    "grad": HistGradientBoostingClassifier(random_state=12)
}

for name in models_full:
    models_full[name].fit(X_train, y_train)
    models_pca[name].fit(X_train_pca, y_train)

print("All models trained.")

All models trained.


In [8]:
from sklearn.metrics import precision_score, recall_score, f1_score

def show_scores(y_true, y_pred):
    print("\tPrecision:", precision_score(y_true=y_true, y_pred=y_pred))
    print("\tRecall:", recall_score(y_true=y_true, y_pred=y_pred))
    print("\tF1:", f1_score(y_true=y_true, y_pred=y_pred))
    print()

def score_model(model, model_name, pca=False):
    print("Scores for", model_name, "(PCA reduced)" if pca else "")
    
    X_eval = X_val_pca if pca else X_val
    y_pred = model.predict(X_eval)
    
    show_scores(y_val, y_pred)

## Scores for preliminary models

In [9]:
full_names = {
    "tree": "Decision Tree Classifier",
    "forest": "Random Forest Classifier",
    "grad": "Gradient Boosting Classifier"
}

results = []

for key, label in full_names.items():
    # Full features
    y_pred_full = models_full[key].predict(X_val)
    print("Scores for", label)
    show_scores(y_val, y_pred_full)
    results.append([
        label,
        "Full",
        precision_score(y_val, y_pred_full),
        recall_score(y_val, y_pred_full),
        f1_score(y_val, y_pred_full)
    ])
    
    # PCA features
    y_pred_pca = models_pca[key].predict(X_val_pca)
    print("Scores for", label, "(PCA reduced)")
    show_scores(y_val, y_pred_pca)
    results.append([
        label,
        "PCA",
        precision_score(y_val, y_pred_pca),
        recall_score(y_val, y_pred_pca),
        f1_score(y_val, y_pred_pca)
    ])

results_df = pd.DataFrame(
    results,
    columns=["Model", "Features", "Precision", "Recall", "F1"]
)

results_df

Scores for Decision Tree Classifier
	Precision: 0.18206079541756168
	Recall: 0.45025756600128786
	F1: 0.25928157589803014

Scores for Decision Tree Classifier (PCA reduced)
	Precision: 0.0946628122707857
	Recall: 0.3531873792659369
	F1: 0.14930756405457823

Scores for Random Forest Classifier
	Precision: 0.20299022426682
	Recall: 0.45460399227301995
	F1: 0.28065990856688533

Scores for Random Forest Classifier (PCA reduced)
	Precision: 0.12061524842314927
	Recall: 0.3509336767546684
	F1: 0.1795272996788273

Scores for Gradient Boosting Classifier
	Precision: 0.09978819486072812
	Recall: 0.8039278815196395
	F1: 0.17753919442568167

Scores for Gradient Boosting Classifier (PCA reduced)
	Precision: 0.046708083279069355
	Recall: 0.8544752092723761
	F1: 0.08857443222587481



,Model,Features,Precision,Recall,F1
0,Decision Tree Classifier,Full,0.182061,0.450258,0.259282
1,Decision Tree Classifier,PCA,0.094663,0.353187,0.149308
2,Random Forest Classifier,Full,0.202990,0.454604,0.280660
3,Random Forest Classifier,PCA,0.120615,0.350934,0.179527
4,Gradient Boosting Classifier,Full,0.099788,0.803928,0.177539
5,Gradient Boosting Classifier,PCA,0.046708,0.854475,0.088574


In [10]:
results_df.sort_values(by="F1", ascending=False)

,Model,Features,Precision,Recall,F1
2,Random Forest Classifier,Full,0.202990,0.454604,0.280660
0,Decision Tree Classifier,Full,0.182061,0.450258,0.259282
3,Random Forest Classifier,PCA,0.120615,0.350934,0.179527
4,Gradient Boosting Classifier,Full,0.099788,0.803928,0.177539
1,Decision Tree Classifier,PCA,0.094663,0.353187,0.149308
5,Gradient Boosting Classifier,PCA,0.046708,0.854475,0.088574


In [11]:
best_row = results_df.sort_values(by="F1", ascending=False).iloc[0]
print("Best preliminary model:")
print(best_row)

Best preliminary model:
Model        Random Forest Classifier
Features                         Full
Precision                     0.20299
Recall                       0.454604
F1                            0.28066
Name: 2, dtype: object
